# Gradient Boosting & XGBoost: Tuning, Importance, and LIME
Accompanies **Unit 6.2: Extreme Gradient Boosting (XGBoost)**.

XGBoost expands the objective function using a second-order Taylor expansion around current predictions:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t^2(\mathbf{x}_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$
where $g_i = \partial_{\hat{y}^{(t-1)}} \ell$ and $h_i = \partial^2_{\hat{y}^{(t-1)}} \ell$.

Hyperparameter Tuning Guidelines:
- `learning_rate` ($\eta$): Step shrinkage factor. Lower values ($0.01-0.05$) prevent early overfitting but require more `n_estimators`.
- `max_depth`: Depth of trees. In boosting, weak shallow trees ($3-6$) prevent individual trees from dominating the ensemble.
- `subsample`: Fraction of rows sampled per tree ($0.6-0.8$ adds stochastic regularization).
- `colsample_bytree`: Fraction of features sampled per tree (decorrelates sequential trees).
- `min_child_weight`: Minimum sum of instance Hessian weights $H_j = \sum h_i$ needed in a leaf. Prevents building leaves on small outlier pockets.
- `gamma` ($\gamma$): Minimum loss reduction gain required to make a split. Acts as a pruning threshold.
- `reg_lambda` ($L_2$) and `reg_alpha` ($L_1$): Regularization on leaf weights.

In this notebook, we tune XGBoost, track sequential deviance, compute permutation importance, and generate a local explanation with LIME.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import log_loss, roc_auc_score

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Staged Deviance: Monitoring Sequential Loss Reduction

In [ ]:
gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

train_loss = [log_loss(y_train, p) for p in gbm.staged_predict_proba(X_train)]
test_loss = [log_loss(y_test, p) for p in gbm.staged_predict_proba(X_test)]

plt.figure(figsize=(8, 4))
plt.plot(train_loss, label='Train Log-Loss', color='royalblue')
plt.plot(test_loss, label='Test Log-Loss', color='crimson')
plt.xlabel('Boosting Iteration', fontsize=11)
plt.ylabel('Negative Log-Likelihood (Deviance)', fontsize=11)
plt.title('Sequential Loss Reduction: Identifying Early Overfitting', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

### Permutation Importance on Test Set

In [ ]:
perm_res = permutation_importance(gbm, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')

sorted_idx = np.argsort(perm_res.importances_mean)[::-1]
print("--- Permutation Feature Importance (Drop in Test ROC-AUC) ---")
for idx in sorted_idx:
    print(f"  {numeric_cols[idx]:15s}: {perm_res.importances_mean[idx]:.4f} +/- {perm_res.importances_std[idx]:.4f}")

### Local Interpretability with LIME
LIME perturbs the features around an individual customer and fits a weighted local linear surrogate model to explain the black-box prediction.

In [ ]:
# Synthetic LIME local explanation demonstration
from sklearn.linear_model import Ridge

target_idx = 0
x_instance = X_test.iloc[target_idx].values

# Generate 500 local Gaussian perturbations around this instance
np.random.seed(42)
perturbations = np.random.normal(loc=x_instance, scale=np.std(X_train.values, axis=0) * 0.2, size=(500, len(x_instance)))
pert_probs = gbm.predict_proba(perturbations)[:, 1]

# Exponential kernel weighting based on Euclidean distance
distances = np.linalg.norm(perturbations - x_instance, axis=1)
weights = np.exp(-(distances ** 2) / (np.mean(distances) ** 2))

# Fit local weighted ridge model
local_surrogate = Ridge(alpha=1.0)
local_surrogate.fit(perturbations, pert_probs, sample_weight=weights)

print(f"Base instance prediction: {gbm.predict_proba(x_instance.reshape(1, -1))[0, 1]:.3f}")
print("\nLocal Linear LIME Weights:")
for col, w in sorted(zip(numeric_cols, local_surrogate.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {col:15s}: {w:+.4f}")